In [ ]:
!pip -q install easyocr openpyxl pandas pillow opencv-python-headless flask flask-cors google-genai

In [ ]:
from google.colab import files
import pandas as pd

uploaded = files.upload()

excel_file = next(
    name for name in uploaded
    if name.lower().endswith(".xlsx")
)

df = pd.read_excel(excel_file)

print("데이터 개수:", len(df))
print("컬럼 이름:", df.columns.tolist())
display(df.head())

In [ ]:
import re

TERM_COL = "어려운 단어"
EXPLAIN_COL = "쉬운 단어"

assert TERM_COL in df.columns, f"열을 찾을 수 없음: {TERM_COL}"
assert EXPLAIN_COL in df.columns, f"열을 찾을 수 없음: {EXPLAIN_COL}"

# 공백 제거
def normalize(text):
    return re.sub(r"\s+", "", str(text)).strip()

# 엑셀 데이터셋을 사전으로 변환
dictionary = {}

for _, row in df.iterrows():
    term = str(row[TERM_COL]).strip()
    explanation = str(row[EXPLAIN_COL]).strip()

    if term and term.lower() != "nan" and explanation.lower() != "nan":
        dictionary[normalize(term)] = {
            "용어": term,
            "쉬운 설명": explanation
        }

# 어려운 단어 검색 함수
def find_matches(ocr_text):
    normalized_ocr = normalize(ocr_text)
    unique_matches = {}

    for key in sorted(dictionary.keys(), key=len, reverse=True):
        if len(key) >= 2 and key in normalized_ocr:
            item = dictionary[key]
            unique_matches[item["용어"]] = item["쉬운 설명"]

    return [
        {"용어": term, "쉬운 설명": explanation}
        for term, explanation in unique_matches.items()
    ]

print("데이터셋 준비 완료!")
print("등록된 어려운 단어 수:", len(dictionary))

In [ ]:
import easyocr
import cv2
import numpy as np
from PIL import Image
from google.colab import files
from IPython.display import display
import os

reader = easyocr.Reader(["ko", "en"], gpu=False)

print("글자가 포함된 실제 이미지 파일(PNG, JPG, JPEG)을 업로드해 주세요.")
uploaded_image = files.upload()

if uploaded_image:
    image_file = next(iter(uploaded_image))
    ext = os.path.splitext(image_file)[1].lower()

    if ext in ['.png', '.jpg', '.jpeg']:
        image = Image.open(image_file).convert("RGB")
        display(image)

        img = np.array(image)
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

        gray = cv2.resize(
            gray, None, fx=1.5, fy=1.5,
            interpolation=cv2.INTER_CUBIC
        )

        results = reader.readtext(gray, detail=1)

        print("\n--- [개별 인식 결과] ---")
        for box, text, confidence in results:
            print(f"{text} (정확도: {confidence:.2f})")

        ocr_text = " ".join(text for box, text, confidence in results)
        full_text = ocr_text

        print("\n--- [전체 인식 결과] ---")
        print(ocr_text)
    else:
        print(f"\n[오류] 업로드된 파일('{image_file}')은 이미지 파일이 아닙니다. 꼭 이미지 파일(.png, .jpg, .jpeg)을 올려주세요.")
else:
    print("업로드된 파일이 없습니다.")

def find_matches(ocr_text):
    normalized_ocr = normalize(ocr_text)
    unique_matches = {}

    for key in sorted(dictionary.keys(), key=len, reverse=True):
        if len(key) >= 2 and key in normalized_ocr:
            item = dictionary[key]
            unique_matches[item["용어"]] = item["쉬운 설명"]

    return [
        {"용어": term, "쉬운 설명": explanation}
        for term, explanation in unique_matches.items()
    ]

In [ ]:
result = find_matches(ocr_text)
print(result)

In [ ]:
import easyocr

try:
    reader
except NameError:
    reader = easyocr.Reader(["ko","en"], gpu = False)

print("OCR 준비 완료")

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode

def take_photo(filename='photo.jpg', quality=0.8):
  js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = 'Capture';
      div.appendChild(capture);

      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
  display(js)
  data = eval_js('takePhoto({})'.format(quality))
  binary = b64decode(data.split(',')[1])
  with open(filename, 'wb') as f:
    f.write(binary)
  return filename

# 웹캠으로 사진 촬영하기
try:
  print("카메라가 켜지면 화면을 보고 'Capture' 버튼을 눌러주세요.")
  image_path = take_photo('webcam_photo.jpg')
  print(f"성공적으로 사진이 촬영되어 '{image_path}'에 저장되었습니다.")

  # 획득한 사진으로 즉시 OCR 인식 수행
  result = reader.readtext(image_path)
  full_text = " ".join([d[1] for d in result])
  print("\n--- [촬영된 사진에서 추출한 텍스트] ---")
  print(full_text)

except Exception as err:
  # 사용자의 기기에 웹캠이 없거나 권한이 없는 경우
  print("카메라 실행에 실패했습니다. 기기에 웹캠이 연결되어 있는지, 브라우저 카메라 권한이 허용되어 있는지 확인해 주세요.")
  print("에러 내용:", err)

In [ ]:
from flask import Flask, request, jsonify
from flask_cors import CORS
from PIL import Image
from io import BytesIO
from google.colab import userdata
from pyngrok import ngrok
from google import genai
import easyocr
import cv2
import numpy as np
import threading
import time

# 1. 기존 OCR 모델 재사용
if "reader" not in globals():
    reader = easyocr.Reader(["ko", "en"], gpu=False)

# 2. API 키 불러오기 (Colab Secrets의 'GOOGLE_API_KEY' 활용)
try:
    api_key = userdata.get("GOOGLE_API_KEY")
except Exception:
    api_key = None

client = genai.Client(api_key=api_key)

# 3. API 서버 생성
app = Flask(__name__)
CORS(app, resources={
    r"/api/*": {
        "origins": [
            "http://localhost:5173",
            "http://localhost:3000"
        ]
    }
})

@app.route("/api/translate", methods=["POST"])
def translate_image():
    try:
        if "image" not in request.files:
            return jsonify({"error": "사진 파일이 없습니다."}), 400

        image_file = request.files["image"]
        image = Image.open(BytesIO(image_file.read())).convert("RGB")

        img = np.array(image)
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        gray = cv2.resize(gray, None, fx=1.5, fy=1.5, interpolation=cv2.INTER_CUBIC)

        results = reader.readtext(gray, detail=1)
        ocr_text = " ".join(text for box, text, confidence in results).strip()

        if not ocr_text:
            return jsonify({"error": "글자를 인식하지 못했어요. 사진을 다시 찍어 주세요."}), 422

        matches = find_matches(ocr_text)
        known_terms = "\n".join(f"- {item['용어']}: {item['쉬운 설명']}" for item in matches)
        if not known_terms:
            known_terms = "일치하는 데이터셋 용어 없음"

        prompt = f"""
당신은 어르신을 위한 쉬운 말 변환 도우미입니다.

[원문]
{ocr_text}

[기존 데이터셋에서 찾은 어려운 단어와 설명]
{known_terms}

다음 기준에 따라 답하세요.
1. 기존 데이터셋에 등록된 단어는 제공된 쉬운 설명을 우선 활용하세요.
2. 데이터셋에 없는 다른 어려운 단어도 찾아 쉬운 말로 설명하세요.
3. 원문의 의미를 유지하면서 전체 문장을 어르신이 이해하기 쉬운 한국어로 바꿔 주세요.
4. 금액, 날짜, 신청 조건 등 중요한 정보는 임의로 바꾸지 마세요.
5. 결과에 어려운 단어 설명과 쉬워진 문장을 구분해서 보여 주세요.
"""
        response = client.models.generate_content(
            model="gemini-2.5-flash-lite",
            contents=prompt
        )

        easy_text = response.text or "번역 결과를 생성하지 못했습니다."

        return jsonify({
            "ocr_text": ocr_text,
            "matches": matches,
            "easy_text": easy_text
        })

    except Exception as e:
        print("번역 처리 오류:", repr(e))
        return jsonify({"error": "사진 처리 또는 번역 중 오류가 발생했습니다."}), 500

@app.route("/health", methods=["GET"])
def health():
    return jsonify({"ok": True})

# 4. 서버 실행
thread = threading.Thread(
    target=lambda: app.run(host="0.0.0.0", port=5000, debug=False, use_reloader=False),
    daemon=True
)
thread.start()
time.sleep(2)

# 5. 외부 접속 URL 발급
try:
    ngrok.set_auth_token(userdata.get("NGROK_AUTHTOKEN"))
    tunnel = ngrok.connect(5000)
    print("API 주소:", tunnel.public_url)
    print("프론트엔드에서 사용할 주소:", tunnel.public_url)
except Exception as e:
    print("ngrok 연결 실패:", e)